In [1]:
# ==============================================================================
# QURAN HOUSE — PHASE 1
# CELL 1: ENVIRONMENT BOOTSTRAP & RUNTIME CONTRACT
# ==============================================================================

import sys
import shutil
import os
import subprocess

# 1. Gerekli kütüphaneleri güvenli içe aktar (Eksikse Colab/Notebook ortamında yükle)
REQUIRED_PACKAGES = {
    "MoviePy": "moviepy",
    "Pillow": "PIL",
    "Requests": "requests"
}

for display_name, import_name in REQUIRED_PACKAGES.items():
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", display_name.lower()])

# 2. Kontrolleri gerçekleştir
results = {}

# Python Sürümü Kontrolü (>= 3.9)
py_version = f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"
results["Python"] = f"PASS ({py_version})" if sys.version_info >= (3, 9) else f"FAIL ({py_version})"

# FFmpeg Varlığı Kontrolü
ffmpeg_path = shutil.which("ffmpeg")
results["FFmpeg"] = "PASS" if ffmpeg_path else "FAIL (not found in PATH)"

# Paket Kontrolleri
for display_name, import_name in REQUIRED_PACKAGES.items():
    try:
        __import__(import_name)
        results[display_name] = "PASS"
    except ImportError:
        results[display_name] = "FAIL"

# PEXELS_API_KEY Kontrolü (Colab Secrets veya Environment)
pexels_key = os.environ.get("PEXELS_API_KEY")
if not pexels_key:
    try:
        from google.colab import userdata
        pexels_key = userdata.get("PEXELS_API_KEY")
    except Exception:
        pass

# Güvenlik Kuralı: Secret değeri ASLA yazdırılmaz, sadece varlığı doğrulanır
results["PEXELS_API_KEY"] = "PASS (configured)" if pexels_key else "FAIL (not configured)"

# Genel Durum
critical_pass = (
    sys.version_info >= (3, 9)
    and ffmpeg_path is not None
    and all("PASS" in results[p] for p in REQUIRED_PACKAGES.keys())
    and "PASS" in results["PEXELS_API_KEY"]
)
status = "PASS" if critical_pass else "FAIL"

# 3. Standart Çıktı Raporu
print("=" * 50)
print("QURAN HOUSE — PHASE 1")
print("CELL 1 — ENVIRONMENT BOOTSTRAP")
print("=" * 50)
for key, val in results.items():
    print(f"{key.ljust(22, '.')} {val}")
print()
print(f"{'Video generation'.ljust(22, '.')} NOT RUN")
print(f"{'Quran API'.ljust(22, '.')} NOT RUN")
print()
print(f"STATUS: {status}")
print("=" * 50)

QURAN HOUSE — PHASE 1
CELL 1 — ENVIRONMENT BOOTSTRAP
Python................ PASS (3.13.15)
FFmpeg................ PASS
MoviePy............... PASS
Pillow................ PASS
Requests.............. PASS
PEXELS_API_KEY........ PASS (configured)

Video generation...... NOT RUN
Quran API............. NOT RUN

STATUS: PASS


In [2]:
# ==============================================================================
# QURAN HOUSE — PHASE 1
# CELL 2: QURAN API CONNECTIVITY & SCHEMA CONTRACT VALIDATION
# ==============================================================================

import time
import requests

QURAN_API_BASE = "https://api.quran.com/api/v4"
TEST_CHAPTER_ID = 112  # Al-Ikhlas

results = {}
critical_pass = False

try:
    # 1. API Bağlantı ve Gecikme Testi
    start_time = time.time()
    response = requests.get(f"{QURAN_API_BASE}/chapters/{TEST_CHAPTER_ID}", timeout=10)
    latency_ms = int((time.time() - start_time) * 1000)

    if response.status_code == 200:
        results["HTTP Connection"] = f"PASS (200 OK)"
        results["Latency"] = f"PASS ({latency_ms} ms)"

        # 2. JSON Ayrıştırma
        data = response.json()
        results["JSON Parsing"] = "PASS"

        # 3. Şema Sözleşmesi Kontrolü (Beklenen alanların varlığı)
        chapter = data.get("chapter", {})
        required_keys = ["id", "name_simple", "name_arabic", "verses_count"]

        if all(k in chapter for k in required_keys) and chapter.get("id") == TEST_CHAPTER_ID:
            name_simple = chapter.get("name_simple")
            results["Schema Contract"] = f"PASS (Chapter {TEST_CHAPTER_ID}: {name_simple})"
            critical_pass = True
        else:
            missing = [k for k in required_keys if k not in chapter]
            results["Schema Contract"] = f"FAIL (missing fields: {missing})"
    else:
        results["HTTP Connection"] = f"FAIL (Status {response.status_code})"
        results["Latency"] = "FAIL"
        results["JSON Parsing"] = "NOT RUN"
        results["Schema Contract"] = "NOT RUN"

except requests.exceptions.RequestException as e:
    results["HTTP Connection"] = f"FAIL ({type(e).__name__})"
    results["Latency"] = "FAIL"
    results["JSON Parsing"] = "NOT RUN"
    results["Schema Contract"] = "NOT RUN"

status = "PASS" if critical_pass else "FAIL"

# 4. Standart Çıktı Raporu
print("=" * 50)
print("QURAN HOUSE — PHASE 1")
print("CELL 2 — QURAN API CONNECTIVITY")
print("=" * 50)
for key, val in results.items():
    print(f"{key.ljust(22, '.')} {val}")
print()
print(f"{'Verse Retrieval'.ljust(22, '.')} NOT RUN")
print(f"{'Audio Retrieval'.ljust(22, '.')} NOT RUN")
print(f"{'Pexels Media'.ljust(22, '.')} NOT RUN")
print(f"{'Video Generation'.ljust(22, '.')} NOT RUN")
print()
print(f"STATUS: {status}")
print("=" * 50)

QURAN HOUSE — PHASE 1
CELL 2 — QURAN API CONNECTIVITY
HTTP Connection....... PASS (200 OK)
Latency............... PASS (273 ms)
JSON Parsing.......... PASS
Schema Contract....... PASS (Chapter 112: Al-Ikhlas)

Verse Retrieval....... NOT RUN
Audio Retrieval....... NOT RUN
Pexels Media.......... NOT RUN
Video Generation...... NOT RUN

STATUS: PASS


In [3]:
# ==============================================================================
# QURAN HOUSE — PHASE 1
# CELL 3: VERSE & TRANSLATION RETRIEVAL (112:1 AL-IKHLAS)
# ==============================================================================

import re
import requests

QURAN_API_BASE = "https://api.quran.com/api/v4"
SURAH_NUMBER = 112
AYAH_NUMBER = 1
VERSE_KEY = f"{SURAH_NUMBER}:{AYAH_NUMBER}"
# Resource ID 20: Saheeh International (Quran Foundation Standard)
TRANSLATION_RESOURCE_ID = 20

results = {}
critical_pass = False
CURRENT_VERSE_DATA = {}

def clean_translation_text(text: str) -> str:
    """Dipnot işaretlerini, <sup> etiketlerini ve HTML kalıntılarını temizler."""
    text = re.sub(r"<sup[^>]*>.*?</sup>", "", text)  # Dipnot numaralarını ve etiketini kaldır
    text = re.sub(r"<[^>]+>", "", text)              # Kalan HTML etiketlerini kaldır
    text = re.sub(r"[˹˺]", "", text)                  # Özel dipnot parantezlerini temizle
    text = re.sub(r"\s+", " ", text).strip()          # Çift boşlukları düzenle
    return text

try:
    # 1. Ayet ve Çeviri Çağrısı (Uthmani text + Translation)
    url = f"{QURAN_API_BASE}/verses/by_key/{VERSE_KEY}"
    params = {
        "language": "en",
        "words": "false",
        "translations": str(TRANSLATION_RESOURCE_ID),
        "fields": "text_uthmani"
    }

    response = requests.get(url, params=params, timeout=10)

    if response.status_code == 200:
        results["API Request"] = "PASS (200 OK)"
        data = response.json().get("verse", {})

        # 2. Uthmani Arapça Metin Doğrulaması
        text_uthmani = data.get("text_uthmani", "").strip()
        if text_uthmani:
            results["Uthmani Arabic"] = f"PASS ({text_uthmani})"
        else:
            results["Uthmani Arabic"] = "FAIL (empty text)"

        # 3. Çeviri Metni Doğrulaması
        translations = data.get("translations", [])
        if translations and len(translations) > 0:
            raw_translation = translations[0].get("text", "")
            cleaned_translation = clean_translation_text(raw_translation)
            results["English Translation"] = f"PASS ({cleaned_translation})"
        else:
            cleaned_translation = ""
            results["English Translation"] = "FAIL (no translation returned)"

        # 4. Bellek Yükü (Payload) Doğrulaması
        if text_uthmani and cleaned_translation:
            CURRENT_VERSE_DATA = {
                "surah_number": SURAH_NUMBER,
                "ayah_number": AYAH_NUMBER,
                "verse_key": VERSE_KEY,
                "text_uthmani": text_uthmani,
                "translation_raw": raw_translation,
                "translation_clean": cleaned_translation,
                "translation_resource_id": TRANSLATION_RESOURCE_ID
            }
            results["Payload Cache"] = "PASS (stored in memory)"
            critical_pass = True
        else:
            results["Payload Cache"] = "FAIL (incomplete data)"
    else:
        results["API Request"] = f"FAIL (HTTP {response.status_code})"
        results["Uthmani Arabic"] = "NOT RUN"
        results["English Translation"] = "NOT RUN"
        results["Payload Cache"] = "NOT RUN"

except requests.exceptions.RequestException as e:
    results["API Request"] = f"FAIL ({type(e).__name__})"
    results["Uthmani Arabic"] = "NOT RUN"
    results["English Translation"] = "NOT RUN"
    results["Payload Cache"] = "NOT RUN"

status = "PASS" if critical_pass else "FAIL"

# 5. Standart Çıktı Raporu
print("=" * 50)
print("QURAN HOUSE — PHASE 1")
print("CELL 3 — VERSE & TRANSLATION RETRIEVAL")
print("=" * 50)
print(f"{'Verse Key'.ljust(22, '.')} {VERSE_KEY} (Al-Ikhlas)")
for key, val in results.items():
    print(f"{key.ljust(22, '.')} {val}")
print()
print(f"{'Audio Retrieval'.ljust(22, '.')} NOT RUN")
print(f"{'Pexels Media'.ljust(22, '.')} NOT RUN")
print(f"{'Video Generation'.ljust(22, '.')} NOT RUN")
print()
print(f"STATUS: {status}")
print("=" * 50)

QURAN HOUSE — PHASE 1
CELL 3 — VERSE & TRANSLATION RETRIEVAL
Verse Key............. 112:1 (Al-Ikhlas)
API Request........... PASS (200 OK)
Uthmani Arabic........ PASS (قُلْ هُوَ ٱللَّهُ أَحَدٌ)
English Translation... PASS (Say, "He is Allāh, [who is] One,)
Payload Cache......... PASS (stored in memory)

Audio Retrieval....... NOT RUN
Pexels Media.......... NOT RUN
Video Generation...... NOT RUN

STATUS: PASS


In [4]:
# ==============================================================================
# QURAN HOUSE — PHASE 1
# CELL 4: RECITATION AUDIO RETRIEVAL & VERIFICATION (API SOURCE OF TRUTH)
# ==============================================================================

import os
import subprocess
import requests

QURAN_API_BASE = "https://api.quran.com/api/v4"
TARGET_RECITER_ID = 7
EXPECTED_RECITER_SUBSTRING = "Mishari Rashid"

# Cell 3'ten gelen veriyi al veya varsayılan 112:1 kullan
verse_key = CURRENT_VERSE_DATA.get("verse_key", "112:1") if "CURRENT_VERSE_DATA" in globals() else "112:1"

AUDIO_DIR = "assets/audio"
os.makedirs(AUDIO_DIR, exist_ok=True)
audio_filename = f"audio_{verse_key.replace(':', '_')}.mp3"
local_audio_path = os.path.join(AUDIO_DIR, audio_filename)

results = {}
critical_pass = False
CURRENT_AUDIO_DATA = {}

def get_audio_duration(file_path: str) -> float:
    """FFprobe kullanarak ses dosyasının kesin süresini (saniye) döndürür."""
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        file_path
    ]
    output = subprocess.check_output(cmd, stderr=subprocess.STDOUT).decode().strip()
    return float(output)

try:
    # 1. KÂRİ KİMLİK DOĞRULAMASI (/resources/recitations)
    rec_res = requests.get(f"{QURAN_API_BASE}/resources/recitations?language=en", timeout=10)
    verified_reciter = None
    if rec_res.status_code == 200:
        rec_list = rec_res.json().get("recitations", [])
        for r in rec_list:
            if r.get("id") == TARGET_RECITER_ID:
                r_name = r.get("reciter_name", "")
                if EXPECTED_RECITER_SUBSTRING.lower() in r_name.lower():
                    verified_reciter = r
                    break

    if verified_reciter:
        verified_name = verified_reciter.get("reciter_name")
        # Quran API meta verisinden tilavet stilini al
        meta_res = requests.get(f"{QURAN_API_BASE}/quran/recitations/{TARGET_RECITER_ID}?verse_key={verse_key}", timeout=10)
        style_info = "Murattal"
        if meta_res.status_code == 200:
            style_info = meta_res.json().get("meta", {}).get("recitation_style", "Murattal")

        results["Reciter Identity"] = f"PASS (API verified: {verified_name}, {style_info})"

        # 2. AYET SES VE SEGMENTLERİNİ SORGULA (/verses/by_key)
        verse_audio_url = f"{QURAN_API_BASE}/verses/by_key/{verse_key}?audio={TARGET_RECITER_ID}"
        audio_lookup_res = requests.get(verse_audio_url, timeout=10)

        if audio_lookup_res.status_code == 200:
            results["API Audio Lookup"] = "PASS (200 OK)"
            audio_info = audio_lookup_res.json().get("verse", {}).get("audio", {})
            rel_url = audio_info.get("url", "")
            segments = audio_info.get("segments", [])

            if rel_url.startswith("http"):
                full_audio_url = rel_url
            elif rel_url.startswith("//"):
                full_audio_url = f"https:{rel_url}"
            else:
                full_audio_url = f"https://verses.quran.com/{rel_url}"

            # 3. SES DOSYASINI İNDİR
            dl_resp = requests.get(full_audio_url, timeout=15)
            if dl_resp.status_code == 200 and len(dl_resp.content) > 0:
                with open(local_audio_path, "wb") as f:
                    f.write(dl_resp.content)
                file_size_kb = os.path.getsize(local_audio_path) / 1024
                results["Audio Download"] = f"PASS ({audio_filename}, {file_size_kb:.1f} KB)"

                # 4. API DURATION / TIMING DOĞRULAMA (Segment bazlı)
                api_duration_sec = 0.0
                if segments and len(segments) > 0:
                    last_end_ms = segments[-1][3]
                    api_duration_sec = last_end_ms / 1000.0
                    results["API Duration"] = f"PASS ({api_duration_sec:.2f} s active recitation)"
                else:
                    results["API Duration"] = "PASS (segments not returned)"

                # 5. FFPROBE CONTAINER DURATION ÖLÇÜMÜ
                try:
                    ffprobe_dur = get_audio_duration(local_audio_path)
                    results["FFprobe Duration"] = f"PASS ({ffprobe_dur:.2f} s total audio)"

                    # 6. SÜRE TUTARLILIĞI KONTROLÜ
                    if api_duration_sec > 0:
                        if 0 <= (ffprobe_dur - api_duration_sec) <= 2.0:
                            results["Duration Consistency"] = "PASS (consistent with segments)"
                        else:
                            results["Duration Consistency"] = "WARN (slight divergence)"
                    else:
                        results["Duration Consistency"] = "PASS (valid container duration)"

                    # 7. BELLEK YÜKÜ (PAYLOAD) KAYDI
                    CURRENT_AUDIO_DATA = {
                        "reciter_id": TARGET_RECITER_ID,
                        "reciter_name": verified_name,
                        "recitation_style": style_info,
                        "verse_key": verse_key,
                        "audio_url": full_audio_url,
                        "local_path": local_audio_path,
                        "duration_sec": ffprobe_dur,
                        "api_active_duration_sec": api_duration_sec,
                        "file_size_kb": file_size_kb
                    }
                    results["Audio Cache"] = "PASS (stored in memory)"
                    critical_pass = True
                except Exception as ex:
                    results["FFprobe Duration"] = f"FAIL (FFprobe error: {ex})"
                    results["Duration Consistency"] = "NOT RUN"
                    results["Audio Cache"] = "NOT RUN"
            else:
                results["Audio Download"] = f"FAIL (HTTP {dl_resp.status_code})"
                results["API Duration"] = "NOT RUN"
                results["FFprobe Duration"] = "NOT RUN"
                results["Duration Consistency"] = "NOT RUN"
                results["Audio Cache"] = "NOT RUN"
        else:
            results["API Audio Lookup"] = f"FAIL (HTTP {audio_lookup_res.status_code})"
            results["Audio Download"] = "NOT RUN"
            results["API Duration"] = "NOT RUN"
            results["FFprobe Duration"] = "NOT RUN"
            results["Duration Consistency"] = "NOT RUN"
            results["Audio Cache"] = "NOT RUN"
    else:
        results["Reciter Identity"] = f"FAIL (Reciter ID {TARGET_RECITER_ID} not verified via API)"
        results["API Audio Lookup"] = "NOT RUN"
        results["Audio Download"] = "NOT RUN"
        results["API Duration"] = "NOT RUN"
        results["FFprobe Duration"] = "NOT RUN"
        results["Duration Consistency"] = "NOT RUN"
        results["Audio Cache"] = "NOT RUN"

except Exception as e:
    results["Reciter Identity"] = f"FAIL ({type(e).__name__})"
    results["API Audio Lookup"] = "NOT RUN"
    results["Audio Download"] = "NOT RUN"
    results["API Duration"] = "NOT RUN"
    results["FFprobe Duration"] = "NOT RUN"
    results["Duration Consistency"] = "NOT RUN"
    results["Audio Cache"] = "NOT RUN"

status = "PASS" if critical_pass else "FAIL"

# 8. Standart Çıktı Raporu
print("=" * 50)
print("QURAN HOUSE — PHASE 1")
print("CELL 4 — RECITATION AUDIO RETRIEVAL (API VERIFIED)")
print("=" * 50)
for key, val in results.items():
    print(f"{key.ljust(22, '.')} {val}")
print()
print(f"{'Pexels Media'.ljust(22, '.')} NOT RUN")
print(f"{'Video Generation'.ljust(22, '.')} NOT RUN")
print()
print(f"STATUS: {status}")
print("=" * 50)

QURAN HOUSE — PHASE 1
CELL 4 — RECITATION AUDIO RETRIEVAL (API VERIFIED)
Reciter Identity...... PASS (API verified: Mishari Rashid al-`Afasy, Murattal)
API Audio Lookup...... PASS (200 OK)
Audio Download........ PASS (audio_112_1.mp3, 47.1 KB)
API Duration.......... PASS (2.30 s active recitation)
FFprobe Duration...... PASS (2.98 s total audio)
Duration Consistency.. PASS (consistent with segments)
Audio Cache........... PASS (stored in memory)

Pexels Media.......... NOT RUN
Video Generation...... NOT RUN

STATUS: PASS


In [5]:
# ==============================================================================
# QURAN HOUSE — PHASE 1
# CELL 5: PEXELS BACKGROUND MEDIA RETRIEVAL & VALIDATION (9:16 PORTRAIT)
# ==============================================================================

import os
import json
import subprocess
import requests

# 1. PEXELS_API_KEY Güvenli Erişimi (Colab Secrets veya Environment)
pexels_key = os.environ.get("PEXELS_API_KEY")
if not pexels_key:
    try:
        from google.colab import userdata
        pexels_key = userdata.get("PEXELS_API_KEY")
    except Exception:
        pass

# 2. Asgari Süre ve Dizin Hazırlığı
min_duration = CURRENT_AUDIO_DATA.get("duration_sec", 2.98) if "CURRENT_AUDIO_DATA" in globals() else 2.98
verse_key = CURRENT_VERSE_DATA.get("verse_key", "112:1") if "CURRENT_VERSE_DATA" in globals() else "112:1"

BG_DIR = "assets/backgrounds"
os.makedirs(BG_DIR, exist_ok=True)
local_bg_path = os.path.join(BG_DIR, f"bg_video_{verse_key.replace(':', '_')}.mp4")

results = {}
critical_pass = False
CURRENT_MEDIA_DATA = {}

def probe_video_stream(file_path: str) -> dict:
    """FFprobe ile video akışının detaylı teknik meta verilerini JSON olarak çeker."""
    cmd = [
        "ffprobe", "-v", "error",
        "-select_streams", "v:0",
        "-show_entries", "stream=width,height,codec_name,duration:format=duration",
        "-of", "json",
        file_path
    ]
    output = subprocess.check_output(cmd).decode().strip()
    data = json.loads(output)
    streams = data.get("streams", [])
    if not streams:
        raise ValueError("No video stream found in file")

    stream = streams[0]
    fmt = data.get("format", {})

    width = int(stream.get("width", 0))
    height = int(stream.get("height", 0))
    codec = stream.get("codec_name", "unknown")
    dur = float(stream.get("duration") or fmt.get("duration") or 0.0)

    return {
        "width": width,
        "height": height,
        "codec": codec,
        "duration_sec": dur,
        "is_vertical": height > width,
        "aspect_ratio": f"{width}:{height}"
    }

if not pexels_key:
    results["PEXELS_API_KEY"] = "FAIL (not configured in environment/secrets)"
else:
    results["PEXELS_API_KEY"] = "PASS (configured)"

    try:
        # 3. Pexels Video Search API Sorgusu (Portrait, Nature/Sky)
        search_url = "https://api.pexels.com/videos/search"
        headers = {"Authorization": pexels_key}
        params = {
            "query": "peaceful nature clouds sky vertical",
            "orientation": "portrait",
            "size": "medium",
            "per_page": 5
        }

        resp = requests.get(search_url, headers=headers, params=params, timeout=12)
        if resp.status_code == 200:
            search_data = resp.json()
            videos = search_data.get("videos", [])
            results["API Video Search"] = f"PASS (found {len(videos)} candidates)"

            # 4. En Uygun 9:16 Video Adayını Belirle
            chosen_video = None
            chosen_file = None

            for v in videos:
                v_dur = v.get("duration", 0)
                # Süre yeterli mi? (veya loop yapılabilir)
                if v_dur >= min_duration:
                    files = v.get("video_files", [])
                    # Dikey MP4 dosyalarını filtrele (öncelik: 1080x1920 veya 720x1280)
                    for vf in sorted(files, key=lambda x: x.get("height", 0), reverse=True):
                        w = vf.get("width", 0)
                        h = vf.get("height", 0)
                        link = vf.get("link", "")
                        if h > w and link:
                            chosen_video = v
                            chosen_file = vf
                            break
                if chosen_video:
                    break

            # İlk tercihte bulunamadıysa ilk dikey adayı al
            if not chosen_video and videos:
                chosen_video = videos[0]
                files = chosen_video.get("video_files", [])
                for vf in files:
                    if vf.get("height", 0) > vf.get("width", 0) and vf.get("link"):
                        chosen_file = vf
                        break

            if chosen_video and chosen_file:
                v_id = chosen_video.get("id")
                target_w = chosen_file.get("width")
                target_h = chosen_file.get("height")
                v_link = chosen_file.get("link")
                results["Candidate Selection"] = f"PASS (ID {v_id}, {target_w}x{target_h})"

                # 5. Video Dosyasını Yerel Ortama İndir
                dl_resp = requests.get(v_link, stream=True, timeout=20)
                if dl_resp.status_code == 200:
                    with open(local_bg_path, "wb") as f:
                        for chunk in dl_resp.iter_content(chunk_size=1024 * 1024):
                            if chunk:
                                f.write(chunk)

                    file_size_mb = os.path.getsize(local_bg_path) / (1024 * 1024)
                    results["Video Download"] = f"PASS ({os.path.basename(local_bg_path)}, {file_size_mb:.2f} MB)"

                    # 6. FFprobe ile Derin Doğrulama
                    try:
                        probe_info = probe_video_stream(local_bg_path)
                        p_w = probe_info["width"]
                        p_h = probe_info["height"]
                        p_dur = probe_info["duration_sec"]
                        p_codec = probe_info["codec"]

                        if probe_info["is_vertical"] and p_dur > 0:
                            results["FFprobe Verification"] = f"PASS ({p_codec}, {p_w}x{p_h}, {p_dur:.1f}s, vertical)"

                            # 7. Bellek Yükü Kaydı (CURRENT_MEDIA_DATA)
                            CURRENT_MEDIA_DATA = {
                                "provider": "pexels",
                                "video_id": v_id,
                                "local_path": local_bg_path,
                                "width": p_w,
                                "height": p_h,
                                "duration_sec": p_dur,
                                "codec": p_codec,
                                "aspect_ratio": probe_info["aspect_ratio"],
                                "file_size_mb": file_size_mb,
                                "source_url": chosen_video.get("url")
                            }
                            results["Media Cache"] = "PASS (stored in memory)"
                            critical_pass = True
                        else:
                            results["FFprobe Verification"] = "FAIL (video is not vertical or invalid duration)"
                            results["Media Cache"] = "NOT RUN"
                    except Exception as ex:
                        results["FFprobe Verification"] = f"FAIL (FFprobe error: {ex})"
                        results["Media Cache"] = "NOT RUN"
                else:
                    results["Video Download"] = f"FAIL (HTTP {dl_resp.status_code})"
                    results["FFprobe Verification"] = "NOT RUN"
                    results["Media Cache"] = "NOT RUN"
            else:
                results["Candidate Selection"] = "FAIL (no vertical MP4 file found in candidates)"
                results["Video Download"] = "NOT RUN"
                results["FFprobe Verification"] = "NOT RUN"
                results["Media Cache"] = "NOT RUN"
        else:
            results["API Video Search"] = f"FAIL (HTTP {resp.status_code})"
            results["Candidate Selection"] = "NOT RUN"
            results["Video Download"] = "NOT RUN"
            results["FFprobe Verification"] = "NOT RUN"
            results["Media Cache"] = "NOT RUN"

    except Exception as e:
        results["API Video Search"] = f"FAIL ({type(e).__name__})"
        results["Candidate Selection"] = "NOT RUN"
        results["Video Download"] = "NOT RUN"
        results["FFprobe Verification"] = "NOT RUN"
        results["Media Cache"] = "NOT RUN"

status = "PASS" if critical_pass else "FAIL"

# 8. Standart Çıktı Raporu
print("=" * 50)
print("QURAN HOUSE — PHASE 1")
print("CELL 5 — PEXELS BACKGROUND MEDIA RETRIEVAL")
print("=" * 50)
for key, val in results.items():
    print(f"{key.ljust(22, '.')} {val}")
print()
print(f"{'Text Overlay'.ljust(22, '.')} NOT RUN")
print(f"{'Video Composition'.ljust(22, '.')} NOT RUN")
print(f"{'Final MP4 Export'.ljust(22, '.')} NOT RUN")
print()
print(f"STATUS: {status}")
print("=" * 50)

QURAN HOUSE — PHASE 1
CELL 5 — PEXELS BACKGROUND MEDIA RETRIEVAL
PEXELS_API_KEY........ PASS (configured)
API Video Search...... PASS (found 5 candidates)
Candidate Selection... PASS (ID 9011099, 1080x1920)
Video Download........ PASS (bg_video_112_1.mp4, 9.21 MB)
FFprobe Verification.. PASS (h264, 1080x1920, 15.2s, vertical)
Media Cache........... PASS (stored in memory)

Text Overlay.......... NOT RUN
Video Composition..... NOT RUN
Final MP4 Export...... NOT RUN

STATUS: PASS


In [6]:
# ==============================================================================
# QURAN HOUSE — PHASE 1
# CELL 6: VIDEO COMPOSITION & PRODUCTION-GRADE REVALIDATION
# ==============================================================================

import os
import json
import subprocess
from PIL import Image, ImageDraw, ImageFont

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_mp4_path = os.path.join(OUTPUT_DIR, "quran_house_112_1.mp4")

results = {}
critical_pass = False

try:
    # 1. PAYLOAD INTEGRITY CHECKS (Zorunlu alan ve dosya varlık kontrolleri)
    verse_req = ["verse_key", "text_uthmani", "translation_clean"]
    audio_req = ["local_path", "duration_sec", "reciter_name"]
    media_req = ["local_path", "width", "height", "duration_sec"]

    v_ok = all(k in CURRENT_VERSE_DATA and CURRENT_VERSE_DATA[k] for k in verse_req)
    a_ok = all(k in CURRENT_AUDIO_DATA and CURRENT_AUDIO_DATA[k] for k in audio_req)
    m_ok = all(k in CURRENT_MEDIA_DATA and CURRENT_MEDIA_DATA[k] for k in media_req)

    audio_path_val = CURRENT_AUDIO_DATA.get("local_path", "")
    bg_path_val = CURRENT_MEDIA_DATA.get("local_path", "")
    audio_file_exists = os.path.exists(audio_path_val)
    media_file_exists = os.path.exists(bg_path_val)

    if v_ok and a_ok and m_ok and audio_file_exists and media_file_exists:
        results["Payload Integrity"] = "PASS"
    else:
        results["Payload Integrity"] = "FAIL (Missing required fields or disk files)"
        raise ValueError("Payload integrity validation failed")

    # 2. SOURCE CONTENT LINKING
    arabic_val = CURRENT_VERSE_DATA["text_uthmani"]
    trans_val = CURRENT_VERSE_DATA["translation_clean"]
    verse_key_val = CURRENT_VERSE_DATA["verse_key"]
    reciter_val = CURRENT_AUDIO_DATA["reciter_name"]
    source_audio_dur = float(CURRENT_AUDIO_DATA["duration_sec"])

    results["Arabic Text"] = f"PASS ({arabic_val})"
    results["English Translation"] = f"PASS ({trans_val})"
    results["Background Linked"] = f"PASS ({os.path.basename(bg_path_val)})"
    results["Audio Linked"] = f"PASS ({os.path.basename(audio_path_val)}, {reciter_val})"

    # 3. TYPOGRAPHY & OVERLAY GENERATION (1080x1920 RGBA)
    W, H = 1080, 1920
    overlay = Image.new("RGBA", (W, H), (0, 0, 0, 0))

    # Metin kontrastı için %42 yarı saydam siyah karartma
    dim_layer = Image.new("RGBA", (W, H), (0, 0, 0, int(255 * 0.42)))
    overlay = Image.alpha_composite(overlay, dim_layer)
    draw = ImageDraw.Draw(overlay)

    # Font yükleyici (Unicode/Arabic fallback destekli)
    def load_best_font(size, bold=False):
        candidates = [
            "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
            "/usr/share/fonts/truetype/freefont/FreeSansBold.ttf" if bold else "/usr/share/fonts/truetype/freefont/FreeSans.ttf"
        ]
        for c in candidates:
            if os.path.exists(c):
                try:
                    return ImageFont.truetype(c, size)
                except Exception:
                    pass
        return ImageFont.load_default()

    font_h  = load_best_font(36, bold=True)
    font_ar = load_best_font(72, bold=True)
    font_tr = load_best_font(44, bold=False)
    font_b  = load_best_font(26, bold=False)

    # Katmanları çiz
    draw.text((W // 2, 280), f"SURAH AL-IKHLAS — {verse_key_val}", font=font_h, fill=(235, 235, 235, 240), anchor="mm")
    draw.text((W // 2, 920), arabic_val, font=font_ar, fill=(255, 255, 255, 255), anchor="mm")
    draw.text((W // 2, 1140), trans_val, font=font_tr, fill=(225, 225, 225, 240), anchor="mm")
    draw.text((W // 2, 1720), "Q U R A N   H O U S E", font=font_b, fill=(180, 180, 180, 180), anchor="mm")

    overlay_png_path = "assets/typography_overlay.png"
    overlay.save(overlay_png_path)
    results["Typography Overlay"] = "PASS"

    # 4. VIDEO COMPOSITION & ENCODING
    render_cmd = [
        "ffmpeg", "-y",
        "-ss", "0", "-t", str(source_audio_dur), "-i", bg_path_val,
        "-i", overlay_png_path,
        "-i", audio_path_val,
        "-filter_complex", "[0:v]scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920[bg];[bg][1:v]overlay=0:0[v]",
        "-map", "[v]",
        "-map", "2:a",
        "-r", "30",
        "-c:v", "libx264", "-preset", "fast", "-crf", "22", "-pix_fmt", "yuv420p",
        "-c:a", "aac", "-b:a", "192k",
        "-shortest",
        output_mp4_path
    ]

    res = subprocess.run(render_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)

    if res.returncode == 0 and os.path.exists(output_mp4_path):
        file_size_mb = os.path.getsize(output_mp4_path) / (1024 * 1024)
        results["Output File"] = f"PASS ({os.path.basename(output_mp4_path)}, {file_size_mb:.2f} MB)"

        # 5. AKIŞ DÜZEYİNDE DERİN FFPROBE DOĞRULAMASI
        probe_cmd = [
            "ffprobe", "-v", "error",
            "-show_entries", "stream=index,codec_type,codec_name,width,height,r_frame_rate,duration:format=duration,size",
            "-of", "json",
            output_mp4_path
        ]
        probe_data = json.loads(subprocess.check_output(probe_cmd).decode())
        streams = probe_data.get("streams", [])
        fmt = probe_data.get("format", {})

        v_stream = next((s for s in streams if s.get("codec_type") == "video"), None)
        a_stream = next((s for s in streams if s.get("codec_type") == "audio"), None)

        # Kontrol: Çözünürlük (1080x1920)
        v_w = int(v_stream.get("width", 0)) if v_stream else 0
        v_h = int(v_stream.get("height", 0)) if v_stream else 0
        results["Video Resolution"] = f"PASS ({v_w}x{v_h})" if (v_w == 1080 and v_h == 1920) else f"FAIL ({v_w}x{v_h})"

        # Kontrol: Gerçek FPS Hesabı
        r_fps_str = v_stream.get("r_frame_rate", "0/1") if v_stream else "0/1"
        num, den = map(float, r_fps_str.split("/")) if "/" in r_fps_str else (float(r_fps_str), 1.0)
        actual_fps = num / den if den > 0 else 0.0
        results["Video FPS"] = f"PASS (~{actual_fps:.0f} fps)" if (29.0 <= actual_fps <= 31.0) else f"WARN ({actual_fps:.2f} fps)"

        # Kontrol: Codec Doğrulamaları
        v_codec = v_stream.get("codec_name", "").upper() if v_stream else "NONE"
        a_codec = a_stream.get("codec_name", "").upper() if a_stream else "NONE"
        results["Video Codec"] = f"PASS ({v_codec})" if ("H264" in v_codec or "AVC" in v_codec) else f"FAIL ({v_codec})"
        results["Audio Codec"] = f"PASS ({a_codec})" if ("AAC" in a_codec) else f"FAIL ({a_codec})"

        # Kontrol: Süreler ve Senkronizasyon Sapması (Duration Delta)
        fmt_dur = float(fmt.get("duration", 0.0))
        v_dur = float(v_stream.get("duration") or fmt_dur) if v_stream else 0.0
        a_dur = float(a_stream.get("duration") or fmt_dur) if a_stream else 0.0
        duration_delta = abs(fmt_dur - source_audio_dur)

        results["Source Audio"] = f"PASS ({source_audio_dur:.2f} s)"
        results["Final Video"] = f"PASS ({v_dur:.2f} s)"
        results["Final Audio"] = f"PASS ({a_dur:.2f} s)"
        results["Duration Delta"] = f"PASS ({duration_delta:.2f} s)" if duration_delta <= 0.25 else f"WARN ({duration_delta:.2f} s)"

        # Genel Kabul Kapısı
        all_ok = (
            "PASS" in results["Payload Integrity"] and
            "PASS" in results["Video Resolution"] and
            "PASS" in results["Video FPS"] and
            "PASS" in results["Video Codec"] and
            "PASS" in results["Audio Codec"] and
            "PASS" in results["Duration Delta"]
        )
        results["FFprobe Verification"] = "PASS" if all_ok else "FAIL"
        critical_pass = all_ok
    else:
        results["Output File"] = "FAIL"
        results["FFprobe Verification"] = "FAIL"

except Exception as e:
    results["Payload Integrity"] = f"FAIL ({type(e).__name__})"
    results["FFprobe Verification"] = "FAIL"

status = "PASS" if critical_pass else "FAIL"

# 6. Standart Raporlama Çıktısı
print("=" * 50)
print("QURAN HOUSE — PHASE 1")
print("CELL 6 — VIDEO COMPOSITION & FINAL VALIDATION")
print("=" * 50)
print(f"{'Payload Integrity'.ljust(23, '.')} {results.get('Payload Integrity', 'FAIL')}")
print(f"{'Arabic Text'.ljust(23, '.')} {results.get('Arabic Text', 'FAIL')}")
print(f"{'English Translation'.ljust(23, '.')} {results.get('English Translation', 'FAIL')}")
print(f"{'Typography Overlay'.ljust(23, '.')} {results.get('Typography Overlay', 'FAIL')}")
print(f"{'Background Linked'.ljust(23, '.')} {results.get('Background Linked', 'FAIL')}")
print(f"{'Audio Linked'.ljust(23, '.')} {results.get('Audio Linked', 'FAIL')}")
print()
print(f"{'Video Resolution'.ljust(23, '.')} {results.get('Video Resolution', 'FAIL')}")
print(f"{'Video FPS'.ljust(23, '.')} {results.get('Video FPS', 'FAIL')}")
print()
print(f"{'Source Audio'.ljust(23, '.')} {results.get('Source Audio', 'FAIL')}")
print(f"{'Final Video'.ljust(23, '.')} {results.get('Final Video', 'FAIL')}")
print(f"{'Final Audio'.ljust(23, '.')} {results.get('Final Audio', 'FAIL')}")
print(f"{'Duration Delta'.ljust(23, '.')} {results.get('Duration Delta', 'FAIL')}")
print()
print(f"{'Audio Codec'.ljust(23, '.')} {results.get('Audio Codec', 'FAIL')}")
print(f"{'Video Codec'.ljust(23, '.')} {results.get('Video Codec', 'FAIL')}")
print()
print(f"{'Output File'.ljust(23, '.')} {results.get('Output File', 'FAIL')}")
print(f"{'FFprobe Verification'.ljust(23, '.')} {results.get('FFprobe Verification', 'FAIL')}")
print()
print(f"STATUS: {status}")
print("=" * 50)

QURAN HOUSE — PHASE 1
CELL 6 — VIDEO COMPOSITION & FINAL VALIDATION
Payload Integrity...... PASS
Arabic Text............ PASS (قُلْ هُوَ ٱللَّهُ أَحَدٌ)
English Translation.... PASS (Say, "He is Allāh, [who is] One,)
Typography Overlay..... PASS
Background Linked...... PASS (bg_video_112_1.mp4)
Audio Linked........... PASS (audio_112_1.mp3, Mishari Rashid al-`Afasy)

Video Resolution....... PASS (1080x1920)
Video FPS.............. PASS (~30 fps)

Source Audio........... PASS (2.98 s)
Final Video............ PASS (2.97 s)
Final Audio............ PASS (2.92 s)
Duration Delta......... PASS (0.01 s)

Audio Codec............ PASS (AAC)
Video Codec............ PASS (H264)

Output File............ PASS (quran_house_112_1.mp4, 1.25 MB)
FFprobe Verification... PASS

STATUS: PASS


In [7]:
from IPython.display import Video

# Oluşturulan video dosyasının yolu
video_path = 'output/quran_house_112_1.mp4'

# Videoyu 400px genişliğinde oynatıcı ile görüntüle
display(Video(video_path, embed=True, width=400))

In [10]:
import time
from IPython.display import display, Javascript

print("Canlı tutma döngüsü başlatıldı...")

while True:
    # Sayfa odağını canlı tutmak için boş bir JS tetikleyicisi
    display(Javascript('console.log("Colab ping atıldı");'))
    time.sleep(60)

Canlı tutma döngüsü başlatıldı...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

KeyboardInterrupt: 